In [ ]:
!pip install -U langchain-huggingface

In [ ]:
# Opening the log file
with open("HDFS_2k.log", "r") as f:
    logs = f.read()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage

from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
import os
from google.colab import userdata

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HF_TOKEN")
llm = HuggingFaceEndpoint(repo_id="meta-llama/Meta-Llama-3-8B-Instruct",task="conversational")

model = ChatHuggingFace(llm=llm)

chat_template=ChatPromptTemplate([
    ('system','You are a {domain} expert'),
    ('human','Can u tell me in detail about the {topic}')
])

prompt= chat_template.invoke({'domain':'cricket','topic':'LBW'})

result=model.invoke(prompt)
print(result.content)

LBW (Leg Before Wicket) is a fascinating topic in cricket, and I'd be happy to dive into the details with you!

**What is LBW?**

LBW is a method of getting a batsman out in cricket, where the umpire rules that the ball would have hit the stumps if the batsman had not been in the way. In other words, the ball would have been "leg before wicket" if the batsman had not taken evasive action.

**How is LBW adjudicated?**

LBW is a complex process, and the umpire must consider several factors to make a decision. Here are the key points to consider:

1. **Ball striking the pad**: The ball must strike the batsman's pad (the area between their legs) in such a way that it would have hit the stumps if the batsman had not been in the way.
2. **The umpire's view**: The umpire must have a clear view of the ball striking the pad and must be able to assess whether it would have hit the stumps.
3. **The batsman's intent**: The umpire must consider whether the batsman was trying to play the ball and wh

# Gemma Version 1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
!pip install -q -U langchain-huggingface langchain-core


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 10.8 MB/s eta 0:00:00


In [5]:
from google.colab import userdata
token = userdata.get("HF_TOKEN")

In [7]:
import torch
from google.colab import userdata
from huggingface_hub import login
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)

llm = HuggingFacePipeline.from_model_id(
    model_id="google/gemma-2-2b-it",
    task="text-generation",
    device_map="auto",
    model_kwargs={"torch_dtype": torch.bfloat16},
    pipeline_kwargs={"max_new_tokens": 400, "do_sample": False, "return_full_text": False},
)
model = ChatHuggingFace(llm=llm)
print("Gemma loaded on:", next(llm.pipeline.model.parameters()).device)


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Gemma loaded on: cpu


In [15]:
import re
from collections import Counter

base = "https://raw.githubusercontent.com/logpai/loghub/master"
!wget -q -O hdfs.log  {base}/HDFS/HDFS_2k.log
!wget -q -O linux.log {base}/Linux/Linux_2k.log
!cat hdfs.log linux.log > sample_production_logs.txt
!echo "$(wc -c < sample_production_logs.txt) bytes  ($(wc -l < sample_production_logs.txt) lines)"


LEVEL_RANK = {
    "TRACE": 0, "DEBUG": 0, "FINE": 0, "INFO": 1, "NOTICE": 1, "INFORMATION": 1,
    "WARN": 3, "WARNING": 3, "ERROR": 6, "ERR": 6, "SEVERE": 6, "FAIL": 6,
    "CRIT": 7, "CRITICAL": 7, "ALERT": 8, "FATAL": 10, "EMERG": 10,
    "EMERGENCY": 10, "PANIC": 10,
}
_LEVEL_RE = re.compile(
    r"\b(TRACE|DEBUG|FINE|INFO(?:RMATION)?|NOTICE|WARN(?:ING)?|ERR(?:OR)?|SEVERE|"
    r"FAIL|CRIT(?:ICAL)?|ALERT|FATAL|EMERG(?:ENCY)?|PANIC)\b")
WARN_THRESHOLD = LEVEL_RANK["WARN"]
_NORM_RE = re.compile(r"\d+|0x[0-9a-f]+|(?:[0-9a-f]{1,3}\.){3}[0-9a-f]{1,3}", re.I)

def template_of(line):
    return _NORM_RE.sub("#", line.lower())

def level_score(line):
    m = _LEVEL_RE.search(line)
    return LEVEL_RANK.get(m.group(1).upper(), 0) if m else None

anomalies = [l.rstrip() for l in open("sample_production_logs.txt")
             if l.strip() and (level_score(l) or 0) >= WARN_THRESHOLD]
print(f"\n{len(anomalies)} anomaly lines found (WARN+).")
print("  by level:", dict(Counter((level_score(l) or 0) for l in anomalies)))


504333 bytes  (3999 lines)

123 anomaly lines found (WARN+).
  by level: {3: 80, 8: 43}


In [ ]:
import re
import json
from langchain_core.messages import HumanMessage

VALID_SEVERITIES = ["INFO", "WARNING", "ERROR", "CRITICAL", "FATAL"]
REQUIRED_KEYS = ["service_name", "timestamp", "error_severity", "suggested_remediation"]
ANOMALY_SEVERITIES = {"WARNING", "ERROR", "CRITICAL", "FATAL"}

SEVERITY_ALIASES = {
    "TRACE": "INFO", "DEBUG": "INFO", "FINE": "INFO", "INFO": "INFO",
    "INFORMATION": "INFO", "NOTICE": "INFO", "WARN": "WARNING", "WARNING": "WARNING",
    "ERROR": "ERROR", "ERR": "ERROR", "SEVERE": "ERROR", "FAIL": "ERROR", "FAILURE": "ERROR",
    "CRIT": "CRITICAL", "CRITICAL": "CRITICAL", "ALERT": "CRITICAL",
    "FATAL": "FATAL", "EMERG": "FATAL", "EMERGENCY": "FATAL", "PANIC": "FATAL",
}
_ALIAS_KEYS_PATTERN = "|".join(sorted(SEVERITY_ALIASES.keys(), key=len, reverse=True))

INSTRUCTIONS = (
    "You are a production log triage engine. Convert the SINGLE log line below into "
    "ONE JSON object and NOTHING else -- no markdown, no code fences, no prose.\n"
    "Exact keys required:\n"
    '  "service_name": the process/daemon that emitted the line (e.g. sshd, logrotate, '
    "DataNode) -- NOT the hostname.\n"
    '  "timestamp": the timestamp exactly as it appears in that line.\n'
    '  "error_severity": EXACTLY one of INFO, WARNING, ERROR, CRITICAL, FATAL.\n'
    '  "suggested_remediation": steps of concrete actions an on-call engineer should take.\n'
    "All values must be non-empty strings."
)

def parse_json_object(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^json\s*", "", text.strip("`"), flags=re.I)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except json.JSONDecodeError:
                return None
    return None

def validate(obj):
    if not isinstance(obj, dict):
        return False, "not an object"
    if set(obj) != set(REQUIRED_KEYS):
        return False, f"keys must be exactly {REQUIRED_KEYS}, got {sorted(obj)}"
    for k in REQUIRED_KEYS:
        if not isinstance(obj[k], str) or not obj[k].strip():
            return False, f"'{k}' must be a non-empty string"
    sev = obj["error_severity"].strip().upper()
    sev = SEVERITY_ALIASES.get(sev, sev)
    if sev not in VALID_SEVERITIES:
        return False, f"bad severity {obj['error_severity']!r}"
    obj["error_severity"] = sev
    return True, ""

def extract_declared_severity(line):
    """
    Return the canonical severity that the line ITSELF declares
    (via a [LEVEL] tag, 'level=LEVEL', a 'LEVEL:' prefix, or the syslog/
    process header), or None if no explicit level field is found.

    Deliberately does NOT scan the free-text message body for severity
    keywords. That's what was causing benign INFO lines whose message
    happens to mention "error" (e.g. "INFO: previous error resolved") to
    get misclassified as ERROR anomalies.
    """
    field_patterns = [
        r'\[(\w+)\]',
        r'\blevel\s*=\s*(\w+)',
        r'\b(' + _ALIAS_KEYS_PATTERN + r')\s*:',
    ]
    for pat in field_patterns:
        m = re.search(pat, line)
        if m:
            cand = m.group(1).upper()
            if cand in SEVERITY_ALIASES:
                return SEVERITY_ALIASES[cand]

    head = line[:50]
    for tok in re.findall(r'[A-Za-z]+', head):
        u = tok.upper()
        if u in SEVERITY_ALIASES:
            return SEVERITY_ALIASES[u]

    return None


def is_anomaly(line):
    """True only if the line's OWN declared level is elevated -- never based
    on keywords appearing inside the message body."""
    sev = extract_declared_severity(line)
    return sev in ANOMALY_SEVERITIES, sev


def template_of(line):
    """Mask variable tokens (numbers, paths) so repeated events of the same
    shape collapse to one template -- one Gemma call per unique event type."""
    t = re.sub(r'\d+', '<NUM>', line)
    t = re.sub(r'(?:/[\w.\-]+)+', '<PATH>', t)
    t = re.sub(r'<NUM>(?:[:\-]<NUM>)+', '<NUM>', t)
    return t


def parse_timestamp(line):
    m = re.match(r'([A-Z][a-z]{2}\s+\d+\s[\d:]{8})', line)
    if m: return m.group(1)
    m = re.match(r'(\d{6}\s\d{6})', line)
    if m: return m.group(1)
    m = re.match(r'(\d{4}-\d{2}-\d{2}[ T][\d:,.]+)', line)
    if m: return m.group(1)
    return line[:19]

def parse_service(line):
    m = re.search(r'[\d:]{8}\s+\S+\s+([A-Za-z][\w.\-]*?)[\(\[:]', line)
    if m: return m.group(1)
    m = re.search(r'(?:INFO|WARN|ERROR|FATAL|DEBUG)\s+([\w.$]+?):', line)
    if m: return m.group(1).split('.')[-1].split('$')[-1]
    return "unknown"

def triage_one(line, retries=2):
    """Ask Gemma for ONE validated record (used to get the remediation text)."""
    reminder = ""
    for attempt in range(retries + 1):
        out = model.invoke([HumanMessage(
            content=INSTRUCTIONS + "\n\nLog line:\n" + line + reminder +
            "\n\nReturn the JSON object now.")]).content
        obj = parse_json_object(out)
        if obj is not None and validate(obj)[0]:
            return obj
        reminder = "\n\nYour last reply was invalid. Return ONLY the JSON object."
    return None

remediation_by_template = {}
records = []
for line in anomalies:
    anomalous, declared_sev = is_anomaly(line)
    if not anomalous:
        continue  # benign line (e.g. INFO message that merely mentions "error") -- skip
    tmpl = template_of(line)
    if tmpl not in remediation_by_template:
        print(f"  asking Gemma about new event type #{len(remediation_by_template) + 1} ...")
        rec = triage_one(line)
        remediation_by_template[tmpl] = rec["suggested_remediation"] if rec else "Investigate this event."
    records.append({
        "service_name": parse_service(line),
        "timestamp": parse_timestamp(line),
        "error_severity": declared_sev,
        "suggested_remediation": remediation_by_template[tmpl],
    })

print(f"\n--- {len(records)} webhook-ready records ---")
print(json.dumps(records, separators=(",", ":")))

[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  asking Gemma about new event type #1 ...


[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  asking Gemma about new event type #2 ...


[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  asking Gemma about new event type #3 ...


# Gemma Version 2

In [ ]:
!pip install -U langchain-huggingface

Requirement already satisfied: langchain-huggingface in /usr/local/lib/python3.12/dist-packages (1.2.2)
Requirement already satisfied: huggingface-hub<2.0.0,>=0.33.4 in /usr/local/lib/python3.12/dist-packages (from langchain-huggingface) (1.19.0)
Requirement already satisfied: langchain-core<2.0.0,>=1.2.31 in /usr/local/lib/python3.12/dist-packages (from langchain-huggingface) (1.4.7)
Requirement already satisfied: tokenizers<1.0.0,>=0.19.1 in /usr/local/lib/python3.12/dist-packages (from langchain-huggingface) (0.22.2)
Requirement already satisfied: click>=8.4.0 in /usr/local/lib/python3.12/dist-packages (from huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (8.4.1)
Requirement already satisfied: filelock>=3.10.0 in /usr/local/lib/python3.12/dist-packages (from huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (3.29.3)
Requirement already satisfied: fsspec>=2023.5.0 in /usr/local/lib/python3.12/dist-packages (from huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (2025.3.0)
Requirement already satisfied: hf-xet<2.0.0,>=1.5.1 in /usr/local/lib/python3.12/dist-packages (from huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (1.5.1)
Requirement already satisfied: httpx<1,>=0.23.0 in /usr/local/lib/python3.12/dist-packages (from huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (0.28.1)
Requirement already satisfied: packaging>=20.9 in /usr/local/lib/python3.12/dist-packages (from huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (26.2)
Requirement already satisfied: pyyaml>=5.1 in /usr/local/lib/python3.12/dist-packages (from huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (6.0.3)
Requirement already satisfied: tqdm>=4.42.1 in /usr/local/lib/python3.12/dist-packages (from huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (4.67.3)
Requirement already satisfied: typer<0.26.0,>=0.20.0 in /usr/local/lib/python3.12/dist-packages (from huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (0.25.1)
Requirement already satisfied: typing-extensions>=4.1.0 in /usr/local/lib/python3.12/dist-packages (from huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (4.15.0)
Requirement already satisfied: jsonpatch<2.0.0,>=1.33.0 in /usr/local/lib/python3.12/dist-packages (from langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (1.33)
Requirement already satisfied: langchain-protocol>=0.0.14 in /usr/local/lib/python3.12/dist-packages (from langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (0.0.17)
Requirement already satisfied: langsmith<1.0.0,>=0.3.45 in /usr/local/lib/python3.12/dist-packages (from langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (0.8.15)
Requirement already satisfied: pydantic<3.0.0,>=2.7.4 in /usr/local/lib/python3.12/dist-packages (from langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (2.12.3)
Requirement already satisfied: tenacity!=8.4.0,<10.0.0,>=8.1.0 in /usr/local/lib/python3.12/dist-packages (from langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (9.1.4)
Requirement already satisfied: uuid-utils<1.0,>=0.12.0 in /usr/local/lib/python3.12/dist-packages (from langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (0.16.0)
Requirement already satisfied: anyio in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.23.0->huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (4.13.0)
Requirement already satisfied: certifi in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.23.0->huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (2026.5.20)
Requirement already satisfied: httpcore==1.* in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.23.0->huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (1.0.9)
Requirement already satisfied: idna in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.23.0->huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (3.18)
Requirement already satisfied: h11>=0.16 in /usr/local/lib/python3.12/dist-packages (from httpcore==1.*->httpx<1,>=0.23.0->huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (0.16.0)
Requirement already satisfied: jsonpointer>=1.9 in /usr/local/lib/python3.12/dist-packages (from jsonpatch<2.0.0,>=1.33.0->langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (3.1.1)
Requirement already satisfied: orjson>=3.9.14 in /usr/local/lib/python3.12/dist-packages (from langsmith<1.0.0,>=0.3.45->langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (3.11.9)
Requirement already satisfied: requests-toolbelt>=1.0.0 in /usr/local/lib/python3.12/dist-packages (from langsmith<1.0.0,>=0.3.45->langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (1.0.0)
Requirement already satisfied: requests>=2.0.0 in /usr/local/lib/python3.12/dist-packages (from langsmith<1.0.0,>=0.3.45->langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (2.32.4)
Requirement already satisfied: websockets>=15.0 in /usr/local/lib/python3.12/dist-packages (from langsmith<1.0.0,>=0.3.45->langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (15.0.1)
Requirement already satisfied: xxhash>=3.0.0 in /usr/local/lib/python3.12/dist-packages (from langsmith<1.0.0,>=0.3.45->langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (3.7.0)
Requirement already satisfied: zstandard>=0.23.0 in /usr/local/lib/python3.12/dist-packages (from langsmith<1.0.0,>=0.3.45->langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (0.25.0)
Requirement already satisfied: annotated-types>=0.6.0 in /usr/local/lib/python3.12/dist-packages (from pydantic<3.0.0,>=2.7.4->langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (0.7.0)
Requirement already satisfied: pydantic-core==2.41.4 in /usr/local/lib/python3.12/dist-packages (from pydantic<3.0.0,>=2.7.4->langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (2.41.4)
Requirement already satisfied: typing-inspection>=0.4.2 in /usr/local/lib/python3.12/dist-packages (from pydantic<3.0.0,>=2.7.4->langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (0.4.2)
Requirement already satisfied: shellingham>=1.3.0 in /usr/local/lib/python3.12/dist-packages (from typer<0.26.0,>=0.20.0->huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (1.5.4)
Requirement already satisfied: rich>=13.8.0 in /usr/local/lib/python3.12/dist-packages (from typer<0.26.0,>=0.20.0->huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (13.9.4)
Requirement already satisfied: annotated-doc>=0.0.2 in /usr/local/lib/python3.12/dist-packages (from typer<0.26.0,>=0.20.0->huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (0.0.4)
Requirement already satisfied: charset_normalizer<4,>=2 in /usr/local/lib/python3.12/dist-packages (from requests>=2.0.0->langsmith<1.0.0,>=0.3.45->langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (3.4.7)
Requirement already satisfied: urllib3<3,>=1.21.1 in /usr/local/lib/python3.12/dist-packages (from requests>=2.0.0->langsmith<1.0.0,>=0.3.45->langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (2.5.0)
Requirement already satisfied: markdown-it-py>=2.2.0 in /usr/local/lib/python3.12/dist-packages (from rich>=13.8.0->typer<0.26.0,>=0.20.0->huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (4.2.0)
Requirement already satisfied: pygments<3.0.0,>=2.13.0 in /usr/local/lib/python3.12/dist-packages (from rich>=13.8.0->typer<0.26.0,>=0.20.0->huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (2.20.0)
Requirement already satisfied: mdurl~=0.1 in /usr/local/lib/python3.12/dist-packages (from markdown-it-py>=2.2.0->rich>=13.8.0->typer<0.26.0,>=0.20.0->huggingface-hub<2.0.0,>=0.33.4->langchain-huggingface) (0.1.2)

In [ ]:
from google.colab import userdata
token = userdata.get("HF_TOKEN")

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)

llm = HuggingFacePipeline.from_model_id(
    model_id="google/gemma-2-2b-it",
    task="text-generation",
    device_map="auto",
    model_kwargs={"torch_dtype": torch.bfloat16},
    pipeline_kwargs={"max_new_tokens": 400, "do_sample": False, "return_full_text": False},
)
model = ChatHuggingFace(llm=llm)
print("Gemma loaded on:", next(llm.pipeline.model.parameters()).device)


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]Downloading (incomplete total...): 0.00B [00:00, ?B/s]Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Gemma loaded on: cpu

In [ ]:
import re
from collections import Counter

base = "https://raw.githubusercontent.com/logpai/loghub/master"
!wget -q -O hdfs.log  {base}/HDFS/HDFS_2k.log
!wget -q -O linux.log {base}/Linux/Linux_2k.log
!cat hdfs.log linux.log > sample_production_logs.txt
!echo "$(wc -c < sample_production_logs.txt) bytes  ($(wc -l < sample_production_logs.txt) lines)"


LEVEL_RANK = {
    "TRACE": 0, "DEBUG": 0, "FINE": 0, "INFO": 1, "NOTICE": 1, "INFORMATION": 1,
    "WARN": 3, "WARNING": 3, "ERROR": 6, "ERR": 6, "SEVERE": 6, "FAIL": 6,
    "CRIT": 7, "CRITICAL": 7, "ALERT": 8, "FATAL": 10, "EMERG": 10,
    "EMERGENCY": 10, "PANIC": 10,
}
_LEVEL_RE = re.compile(
    r"\b(TRACE|DEBUG|FINE|INFO(?:RMATION)?|NOTICE|WARN(?:ING)?|ERR(?:OR)?|SEVERE|"
    r"FAIL|CRIT(?:ICAL)?|ALERT|FATAL|EMERG(?:ENCY)?|PANIC)\b")
WARN_THRESHOLD = LEVEL_RANK["WARN"]
_NORM_RE = re.compile(r"\d+|0x[0-9a-f]+|(?:[0-9a-f]{1,3}\.){3}[0-9a-f]{1,3}", re.I)

def template_of(line):
    return _NORM_RE.sub("#", line.lower())

def level_score(line):
    m = _LEVEL_RE.search(line)
    return LEVEL_RANK.get(m.group(1).upper(), 0) if m else None

anomalies = [l.rstrip() for l in open("sample_production_logs.txt")
             if l.strip() and (level_score(l) or 0) >= WARN_THRESHOLD]
print(f"\n{len(anomalies)} anomaly lines found (WARN+).")
print("  by level:", dict(Counter((level_score(l) or 0) for l in anomalies)))


504333 bytes  (3999 lines)

123 anomaly lines found (WARN+).
  by level: {3: 80, 8: 43}

In [ ]:
import json
from langchain_core.messages import HumanMessage

VALID_SEVERITIES = ["INFO", "WARNING", "ERROR", "CRITICAL", "FATAL"]
REQUIRED_KEYS = ["service_name", "timestamp", "error_severity", "suggested_remediation"]

SEVERITY_ALIASES = {
    "TRACE": "INFO", "DEBUG": "INFO", "FINE": "INFO", "INFO": "INFO",
    "INFORMATION": "INFO", "NOTICE": "INFO", "WARN": "WARNING", "WARNING": "WARNING",
    "ERROR": "ERROR", "ERR": "ERROR", "SEVERE": "ERROR", "FAIL": "ERROR", "FAILURE": "ERROR",
    "CRIT": "CRITICAL", "CRITICAL": "CRITICAL", "ALERT": "CRITICAL",
    "FATAL": "FATAL", "EMERG": "FATAL", "EMERGENCY": "FATAL", "PANIC": "FATAL",
}

INSTRUCTIONS = (
    "You are a production log triage engine. Convert the SINGLE log line below into "
    "ONE JSON object and NOTHING else -- no markdown, no code fences, no prose.\n"
    "Exact keys required:\n"
    '  "service_name": the process/daemon that emitted the line (e.g. sshd, logrotate, '
    "DataNode) -- NOT the hostname.\n"
    '  "timestamp": the timestamp exactly as it appears in that line.\n'
    '  "error_severity": EXACTLY one of INFO, WARNING, ERROR, CRITICAL, FATAL.\n'
    '  "suggested_remediation": one concrete action an on-call engineer should take.\n'
    "All values must be non-empty strings."
)

def parse_json_object(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^json\s*", "", text.strip("`"), flags=re.I)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except json.JSONDecodeError:
                return None
    return None

def validate(obj):
    if not isinstance(obj, dict):
        return False, "not an object"
    if set(obj) != set(REQUIRED_KEYS):
        return False, f"keys must be exactly {REQUIRED_KEYS}, got {sorted(obj)}"
    for k in REQUIRED_KEYS:
        if not isinstance(obj[k], str) or not obj[k].strip():
            return False, f"'{k}' must be a non-empty string"
    sev = obj["error_severity"].strip().upper()
    sev = SEVERITY_ALIASES.get(sev, sev)
    if sev not in VALID_SEVERITIES:
        return False, f"bad severity {obj['error_severity']!r}"
    obj["error_severity"] = sev
    return True, ""

def parse_timestamp(line):
    m = re.match(r'([A-Z][a-z]{2}\s+\d+\s[\d:]{8})', line)
    if m: return m.group(1)
    m = re.match(r'(\d{6}\s\d{6})', line)
    if m: return m.group(1)
    m = re.match(r'(\d{4}-\d{2}-\d{2}[ T][\d:,.]+)', line)
    if m: return m.group(1)
    return line[:19]

def parse_service(line):
    m = re.search(r'[\d:]{8}\s+\S+\s+([A-Za-z][\w.\-]*?)[\(\[:]', line)
    if m: return m.group(1)
    m = re.search(r'(?:INFO|WARN|ERROR|FATAL|DEBUG)\s+([\w.$]+?):', line)
    if m: return m.group(1).split('.')[-1].split('$')[-1]
    return "unknown"

def triage_one(line, retries=2):
    """Ask Gemma for ONE validated record (used to get the remediation text)."""
    reminder = ""
    for attempt in range(retries + 1):
        out = model.invoke([HumanMessage(
            content=INSTRUCTIONS + "\n\nLog line:\n" + line + reminder +
            "\n\nReturn the JSON object now.")]).content
        obj = parse_json_object(out)
        if obj is not None and validate(obj)[0]:
            return obj
        reminder = "\n\nYour last reply was invalid. Return ONLY the JSON object."
    return None

SEV_NAME = {3: "WARNING", 6: "ERROR", 7: "CRITICAL", 8: "CRITICAL", 10: "FATAL"}
remediation_by_template = {}
records = []
for line in anomalies:
    tmpl = template_of(line)
    if tmpl not in remediation_by_template:
        print(f"  asking Gemma about new event type #{len(remediation_by_template) + 1} ...")
        rec = triage_one(line)
        remediation_by_template[tmpl] = rec["suggested_remediation"] if rec else "Investigate this event."
    records.append({
        "service_name": parse_service(line),
        "timestamp": parse_timestamp(line),
        "error_severity": SEV_NAME[level_score(line)],
        "suggested_remediation": remediation_by_template[tmpl],
    })

print(f"\n--- {len(records)} webhook-ready records ---")
print(json.dumps(records, separators=(",", ":")))


[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  asking Gemma about new event type #1 ...
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  asking Gemma about new event type #2 ...
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  asking Gemma about new event type #3 ...
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  asking Gemma about new event type #4 ...
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  asking Gemma about new event type #5 ...

--- 123 webhook-ready records ---
[{"service_name":"DataXceiver","timestamp":"081109 214043","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081109 214402","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 214529","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 214910","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 215136","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 215259","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081109 215702","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 220032","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081109 220403","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081109 221105","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081109 221151","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 222040","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081109 222041","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081109 222342","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 222650","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 223910","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 224054","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 224234","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 224420","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 224741","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081109 230110","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 023456","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 030331","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 030942","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 032126","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 070326","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 071657","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 072124","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 080546","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 080847","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 081044","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 081054","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 081337","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 081515","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 081643","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 081741","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 082702","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 082706","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 082737","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 082954","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 083045","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 091216","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 091550","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 091657","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 092151","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 093020","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 094019","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 124159","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 124817","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 125138","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 125914","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 125920","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 130315","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 131041","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 131233","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 131806","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 131950","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 132523","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 133657","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 133946","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 135429","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 135653","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 135806","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 163945","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 170322","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 175354","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 175818","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 175934","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 191403","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 192015","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 193334","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 194159","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 194735","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 231904","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081110 231911","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081110 233954","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081111 004102","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the data transfer failure."},{"service_name":"DataXceiver","timestamp":"081111 011254","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081111 012254","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"DataXceiver","timestamp":"081111 014431","error_severity":"WARNING","suggested_remediation":"Investigate the exception and address the issue causing the failure."},{"service_name":"logrotate","timestamp":"Jun 15 04:06:20","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 16 04:10:24","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 17 04:03:36","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 18 04:07:06","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 19 04:09:11","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 20 04:02:55","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 21 04:06:59","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 22 04:06:00","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 23 04:05:30","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 24 04:05:35","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 25 04:04:26","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 26 04:04:31","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 27 04:02:49","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 28 04:03:17","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 29 04:03:12","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jun 30 04:03:43","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul  1 04:05:19","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul  2 04:04:03","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul  3 04:08:03","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul  4 04:03:08","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul  5 04:03:18","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul  6 04:03:10","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul  7 04:04:33","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul  8 04:04:20","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul  9 04:04:25","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 10 04:04:46","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 11 04:03:05","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 12 04:03:42","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 13 04:04:45","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 14 04:08:25","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 15 04:05:03","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 16 04:07:34","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 17 04:08:23","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 18 04:03:25","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 19 04:03:45","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 20 04:05:04","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 21 04:11:28","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 22 04:07:47","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 23 04:09:37","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 24 04:20:42","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 25 04:04:00","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 26 04:05:24","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."},{"service_name":"logrotate","timestamp":"Jul 27 04:16:09","error_severity":"CRITICAL","suggested_remediation":"Investigate logrotate logs for details and address the issue."}]